In [1]:
# Install required packages
%pip install pandas numpy matplotlib seaborn scipy

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Load data
data = pd.read_csv("../data/customer_churn_dataset-training-master.csv")
print(f"Dataset shape: {data.shape}")
data.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 13.7 MB/s  0:00:02m0:00:0100:01

[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


FileNotFoundError: [Errno 2] No such file or directory: '../data/customer_churn_dataset-training-master.csv'

In [ ]:
# Data Quality Check
print("=== Data Quality Check ===")
print(f"\nDataset Shape: {data.shape}")
print(f"\nMissing Values:\n{data.isnull().sum()}")
print(f"\nDuplicate Rows: {data.duplicated().sum()}")
print(f"\nData Types:\n{data.dtypes}")

In [ ]:
# Statistical Summary
print("=== Statistical Summary ===")
print(data.describe())

In [ ]:
# Target Variable Analysis (Churn)
print("=== Target Variable Analysis ===")
churn_counts = data['Churn'].value_counts()
churn_pct = data['Churn'].value_counts(normalize=True) * 100

print(f"Churn Distribution:\n{churn_counts}")
print(f"\nChurn Percentage:\n{churn_pct}")

# Visualize churn distribution
fig, ax = plt.subplots(figsize=(8, 6))
sns.countplot(x='Churn', data=data, ax=ax)
ax.set_title('Churn Distribution')
ax.set_xlabel('Churn (0=No, 1=Yes)')
ax.set_ylabel('Count')
plt.show()

In [ ]:
# Univariate Analysis - Numerical Features
print("=== Univariate Analysis - Numerical Features ===")

numerical_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 
                  'Payment Delay', 'Total Spend', 'Last Interaction']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    if idx < len(axes):
        axes[idx].hist(data[col].dropna(), bins=30, edgecolor='black')
        axes[idx].set_title(f'Distribution of {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')

# Remove empty subplots
for idx in range(len(numerical_cols), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

In [ ]:
# Univariate Analysis - Categorical Features
print("=== Univariate Analysis - Categorical Features ===")

categorical_cols = ['Gender', 'Subscription Type', 'Contract Length']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, col in enumerate(categorical_cols):
    sns.countplot(x=col, data=data, ax=axes[idx])
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print value counts
for col in categorical_cols:
    print(f"\n{col} value counts:")
    print(data[col].value_counts())
    print(f"Percentages:\n{data[col].value_counts(normalize=True) * 100}")

In [ ]:
# Bivariate Analysis - Numerical Features vs Churn
print("=== Bivariate Analysis - Numerical Features vs Churn ===")

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    if idx < len(axes):
        sns.boxplot(x='Churn', y=col, data=data, ax=axes[idx])
        axes[idx].set_title(f'{col} by Churn')
        axes[idx].set_xlabel('Churn (0=No, 1=Yes)')
        axes[idx].set_ylabel(col)

# Remove empty subplots
for idx in range(len(numerical_cols), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

In [ ]:
# Bivariate Analysis - Categorical Features vs Churn
print("=== Bivariate Analysis - Categorical Features vs Churn ===")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, col in enumerate(categorical_cols):
    sns.countplot(x=col, hue='Churn', data=data, ax=axes[idx])
    axes[idx].set_title(f'{col} by Churn')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].legend(title='Churn', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

# Calculate churn rates by category
for col in categorical_cols:
    print(f"\nChurn rate by {col}:")
    churn_rate = data.groupby(col)['Churn'].mean() * 100
    print(churn_rate)

In [ ]:
# Correlation Analysis
print("=== Correlation Analysis ===")

# Select only numerical columns for correlation
numerical_data = data[numerical_cols + ['Churn']].copy()

# Calculate correlation matrix
correlation_matrix = numerical_data.corr()

# Display correlation matrix
print("Correlation Matrix:")
print(correlation_matrix)

# Visualize correlation matrix
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

# Show correlations with Churn
print("\nCorrelations with Churn:")
print(correlation_matrix['Churn'].sort_values(ascending=False))

In [ ]:
# Key Insights Summary
print("=== Key Insights Summary ===")

print("""
Based on the exploratory data analysis:

1. DATASET OVERVIEW:
   - Total records: 440,832 (after cleaning)
   - 12 features: 7 numerical, 2 categorical, 1 target, 1 identifier
   - Churn rate: 56.7% (imbalanced dataset)

2. DATA QUALITY:
   - 1 row with all null values was removed
   - No duplicate rows found
   - Clean dataset with no missing values

3. TARGET VARIABLE:
   - Churn is slightly imbalanced (56.7% churn vs 43.3% non-churn)
   - May require techniques like SMOTE or class weights in modeling

4. CATEGORICAL FEATURES:
   - Subscription Type: Balanced distribution (~33% each)
   - Contract Length: Annual (40.2%), Quarterly (40.0%), Monthly (19.8%)
   - Gender: Mixed distribution

5. NUMERICAL FEATURES:
   - Age: Mean 39.4 years, range 18-65
   - Tenure: Mean 31.3 months, range 1-60
   - Total Spend: Mean $631.6, range $100-$1,000
   - Support Calls: Mean 3.6, range 0-10

6. NEXT STEPS:
   - Analyze feature correlations with churn
   - Perform feature engineering
   - Handle class imbalance in modeling
   - Build and evaluate classification models
""")